In [1]:
# NativeNext management setup; original course cells follow with the explicitly documented batch-parameter override.
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261021T122137-ceadc417')
import next_identity as _identity
_identity.write_child_runtime_identity()
import next_adapter as _next
_next.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261021T122137-ceadc417\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261021T122137-ceadc417','04_fastai_part2/11_initializing.ipynb')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Full training can exceed a free Colab session; QUICK_RUN does not shorten these original Part 2 experiments.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/11_initializing.ipynb

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '11_initializing.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hi\20261021T122137-ceadc417\source\HaiLong_Colab_Exercises\04_fastai_part2
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hi\20261021T122137-ceadc417\source\HaiLong_Colab_Exercises\04_fastai_part2\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
MAX_BATCH_SIZE = 1024  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

In [4]:
#|default_exp init

# Initialization

In [5]:
#|export
import pickle,gzip,math,os,time,shutil,torch,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
import sys,gc,traceback
import fastcore.all as fc
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter,itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager

import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *

In [6]:
from fastcore.test import test_close

torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)

import logging
logging.disable(logging.WARNING)

set_seed(42)

In [7]:
xl,yl = 'image','label'
name = "zalando-datasets/fashion_mnist"
dsd = load_dataset(name)

@inplace
def transformi(b): b[xl] = [TF.to_tensor(o) for o in b[xl]]

bs = min(1024, MAX_BATCH_SIZE)
tds = dsd.with_transform(transformi)

dls = DataLoaders.from_dd(tds, bs, num_workers=4)
dt = dls.train
xb,yb = next(iter(dt))
xb.shape,yb[:10]

README.md: 0.00B [00:00, ?B/s]

C:\Users\[LOCAL_USER]\.v\hlimg31\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hi\20261021T122137-ceadc417\private_cache\hf\hub\datasets--zalando-datasets--fashion_mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/30.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

(torch.Size([1024, 1, 28, 28]), tensor([5, 7, 4, 7, 3, 8, 9, 5, 3, 1]))

In [8]:
def get_model():
    return nn.Sequential(conv(1 ,8), conv(8 ,16), conv(16,32), conv(32,64),
                         conv(64,10, act=False), nn.Flatten()).to(def_device)

In [9]:
MomentumLearner(get_model(), dls, F.cross_entropy, cbs=[DeviceCB()]).lr_find(gamma=1.1, start_lr=1e-2)

In [10]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(fc.risinstance(nn.ReLU))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
learn = MomentumLearner(get_model(), dls, F.cross_entropy, lr=0.2, cbs=cbs)

In [11]:
learn.fit(1)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 509.147 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.147 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.256,2.033,0,train
0.631,0.962,0,eval


Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.147 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.147 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.147 GB. Budget is not Drive quota.


In [12]:
astats.color_dim()

In [13]:
astats.plot_stats()

In [14]:
#|export
def clean_ipython_hist():
    # Code in this function mainly copied from IPython source
    if not 'get_ipython' in globals(): return
    ip = get_ipython()
    user_ns = ip.user_ns
    ip.displayhook.flush()
    pc = ip.displayhook.prompt_count + 1
    for n in range(1, pc): user_ns.pop('_i'+repr(n),None)
    user_ns.update(dict(_i='',_ii='',_iii=''))
    hm = ip.history_manager
    hm.input_hist_parsed[:] = [''] * pc
    hm.input_hist_raw[:] = [''] * pc
    hm._i = hm._ii = hm._iii = hm._i00 =  ''

In [15]:
#|export
def clean_tb():
    # h/t Piotr Czapla
    if hasattr(sys, 'last_traceback'):
        traceback.clear_frames(sys.last_traceback)
        delattr(sys, 'last_traceback')
    if hasattr(sys, 'last_type'): delattr(sys, 'last_type')
    if hasattr(sys, 'last_value'): delattr(sys, 'last_value')

In [16]:
#|export
def clean_mem():
    clean_tb()
    clean_ipython_hist()
    gc.collect()
    torch.cuda.empty_cache()

## Glorot/Xavier init

In [17]:
x = torch.randn(200, 100)
for i in range(50): x = x @ torch.randn(100,100)
x[0:5,0:5]

tensor([[nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan]])

The result is `nan`s everywhere. So maybe the scale of our matrix was too big, and we need to have smaller weights? But if we use too small weights, we will have the opposite problem—the scale of our activations will go from 1 to 0.1, and after 50 layers we'll be left with zeros everywhere:

In [18]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.01)
x[0:5,0:5]

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

So we have to scale our weight matrices exactly right so that the standard deviation of our activations stays at 1. We can compute the exact value to use mathematically, as illustrated by Xavier Glorot and Yoshua Bengio in ["Understanding the Difficulty of Training Deep Feedforward Neural Networks"](http://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf). The right scale for a given layer is $1/\sqrt{n_{in}}$, where $n_{in}$ represents the number of inputs.

In [19]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.1)
x[0:5,0:5]

tensor([[ 1.33, -1.54,  0.54,  0.13,  3.51],
        [ 0.20,  0.95,  0.61, -0.02,  1.73],
        [-0.69,  1.72,  0.36, -0.64, -2.40],
        [ 0.03,  0.73,  0.40, -1.24, -0.64],
        [ 0.33, -0.33,  0.03, -0.32,  0.62]])

### Background

### Variance and standard deviation

Variance is the average of how far away each data point is from the mean. E.g.:

In [20]:
t = torch.tensor([1.,2.,4.,18])

In [21]:
m = t.mean(); m

tensor(6.25)

In [22]:
(t-m).mean()

tensor(0.)

Oops. We can't do that. Because by definition the positives and negatives cancel out. So we can fix that in one of (at least) two ways:

In [23]:
(t-m).pow(2).mean()

tensor(47.19)

In [24]:
(t-m).abs().mean()

tensor(5.88)

But the first of these is now a totally different scale, since we squared. So let's undo that at the end.

In [25]:
(t-m).pow(2).mean().sqrt()

tensor(6.87)

They're still different. Why?

Note that we have one outlier (`18`). In the version where we square everything, it makes that much bigger than everything else.

`(t-m).pow(2).mean()` is refered to as **variance**. It's a measure of how spread out the data is, and is particularly sensitive to outliers.

When we take the sqrt of the variance, we get the **standard deviation**. Since it's on the same kind of scale as the original data, it's generally more interpretable. However, since `sqrt(1)==1`, it doesn't much matter which we use when talking about *unit variance* for initializing neural nets.

The standard deviation represents if the data stays close to the mean or on the contrary gets values that are far away. It's computed by the following formula:

$$\sigma = \sqrt{\frac{1}{n}\left[(x_{0}-m)^{2} + (x_{1}-m)^{2} + \cdots + (x_{n-1}-m)^{2}\right]}$$

where m is the mean and $\sigma$ (the greek letter sigma) is the standard deviation. Here we have a mean of 0, so it's just the square root of the mean of x squared.

`(t-m).abs().mean()` is referred to as the **mean absolute deviation**. It isn't used nearly as much as it deserves to be, because mathematicians don't like how awkward it is to work with. But that shouldn't stop us, because we have computers and stuff.

Here's a useful thing to note about variance:

In [26]:
(t-m).pow(2).mean(), (t*t).mean() - (m*m)

(tensor(47.19), tensor(47.19))

You can see why these are equal if you want to work thru the algebra. Or not.

But, what's important here is that the latter is generally much easier to work with. In particular, you only have to track two things: the sum of the data, and the sum of squares of the data. Whereas in the first form you actually have to go thru all the data twice (once to calculate the mean, once to calculate the differences).

Let's go steal the LaTeX from [Wikipedia](https://en.wikipedia.org/wiki/Variance):

$$\operatorname{E}\left[X^2 \right] - \operatorname{E}[X]^2$$

### Covariance

Here's how Wikipedia defines covariance:

$$\operatorname{cov}(X,Y) = \operatorname{E}{\big[(X - \operatorname{E}[X])(Y - \operatorname{E}[Y])\big]}$$

Let's see that in code. So now we need two vectors.

In [27]:
t

tensor([ 1.,  2.,  4., 18.])

In [28]:
# `u` is twice `t`, plus a bit of randomness
u = t*2
u *= torch.randn_like(t)/10+0.95

plt.scatter(t, u);

In [29]:
prod = (t-t.mean())*(u-u.mean()); prod

tensor([ 51.67,  35.81,  11.57, 275.08])

In [30]:
prod.mean()

tensor(93.53)

In [31]:
v = torch.randn_like(t)
plt.scatter(t, v);

In [32]:
((t-t.mean())*(v-v.mean())).mean()

tensor(2.91)

It's generally more conveniently defined like so:

$$\operatorname{E}\left[X Y\right] - \operatorname{E}\left[X\right] \operatorname{E}\left[Y\right]$$

In [33]:
cov = (t*v).mean() - t.mean()*v.mean(); cov

tensor(2.91)

From now on, you're not allowed to look at an equation (or especially type it in LaTeX) without also typing it in Python and actually calculating some values. Ideally, you should also plot some values.

Finally, here is the Pearson correlation coefficient:

$$\rho_{X,Y}= \frac{\operatorname{cov}(X,Y)}{\sigma_X \sigma_Y}$$

In [34]:
cov / (t.std() * v.std())

tensor(0.34)

It's just a scaled version of the same thing.

In [35]:
x.std()

tensor(1.13)

### Xavier init derivation

When we do `y = a @ x`, the coefficients of `y` are defined by

$$y_{i} = a_{i,0} x_{0} + a_{i,1} x_{1} + \cdots + a_{i,n-1} x_{n-1} = \sum_{k=0}^{n-1} a_{i,k} x_{k}$$

or in pure python code:
```
y[i] = sum([c*d for c,d in zip(a[i], x)])
```

or in numpy/pytorch code:
```
y[i] = (a[i]*x).sum()
```

At the very beginning, our `x` vector has a mean of roughly 0. and a standard deviation of roughly 1. (since we picked it that way).

In [36]:
x = torch.randn(100)
x.mean(), x.std()

(tensor(0.19), tensor(0.96))

If we go back to `y = a @ x` and assume that we chose weights for `a` that also have a mean of 0, we can compute the standard deviation of `y` quite easily. Since it's random, and we may fall on bad numbers, we repeat the operation 100 times.

In [37]:
mean,sqr = 0.,0.
for i in range(100):
    x = torch.randn(100)
    a = torch.randn(512, 100)
    y = a @ x
    mean += y.mean().item()
    sqr  += y.pow(2).mean().item()
mean/100,sqr/100

(-0.021637059338390826, 101.61874794006347)

Now that looks very close to the dimension of our matrix 100. And that's no coincidence! When you compute y, you sum 100 product of one element of a by one element of x. So what's the mean and the standard deviation of such a product? We can show mathematically that as long as the elements in `a` and the elements in `x` are independent, the mean is 0 and the std is 1. This can also be seen experimentally:

In [38]:
mean,sqr = 0.,0.
for i in range(10000):
    x = torch.randn(1)
    a = torch.randn(1)
    y = a*x
    mean += y.item()
    sqr  += y.pow(2).item()
mean/10000,sqr/10000

(-0.003184134499988704, 0.9690546516017775)

Then we sum 100 of those things that have a mean of zero, and a mean of squares of 1, so we get something that has a mean of 0, and mean of square of 100, hence `math.sqrt(100)` being our magic number. If we scale the weights of the matrix and divide them by this `math.sqrt(100)`, it will give us a `y` of scale 1, and repeating the product has many times as we want won't overflow or vanish.

## Kaiming/He init

("He" is a Chinese surname and is pronouced like "Her", not like "Hee".)

### Background

In [39]:
x = torch.randn(200, 100)
y = torch.randn(200)

In [40]:
from math import sqrt

In [41]:
w1 = torch.randn(100,50) / sqrt(100)
b1 = torch.zeros(50)
w2 = torch.randn(50,1) / sqrt(50)
b2 = torch.zeros(1)

In [42]:
def lin(x, w, b): return x @ w + b

In [43]:
l1 = lin(x, w1, b1)
l1.mean(),l1.std()

(tensor(0.01), tensor(0.98))

In [44]:
def relu(x): return x.clamp_min(0.)

In [45]:
l2 = relu(l1)
l2.mean(),l2.std()

(tensor(0.40), tensor(0.58))

In [46]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * 0.1))
x[0:5,0:5]

tensor([[    0.00,     0.00,     0.00,     0.00,     0.00],
        [    0.00,     0.00,     0.00,     0.00,     0.00],
        [    0.00,     0.00,     0.00,     0.00,     0.00],
        [    0.00,     0.00,     0.00,     0.00,     0.00],
        [    0.00,     0.00,     0.00,     0.00,     0.00]])

In ["Delving Deep into Rectifiers: Surpassing Human-Level Performance"](https://arxiv.org/abs/1502.01852) Kaiming He et al. show that we should use the following scale instead: $\sqrt{2 / n_{in}}$, where $n_{in}$ is the number of inputs of our model.

In [47]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * sqrt(2/100)))
x[0:5,0:5]

tensor([[0.40, 0.00, 0.00, 0.28, 0.00],
        [0.37, 0.00, 0.00, 0.30, 0.00],
        [0.28, 0.00, 0.00, 0.20, 0.00],
        [0.36, 0.00, 0.00, 0.30, 0.00],
        [0.25, 0.00, 0.00, 0.19, 0.00]])

### Applying an init function

In [48]:
model = get_model()
model.apply(lambda m: print(type(m).__name__));

Conv2d
ReLU
Sequential
Conv2d
ReLU
Sequential
Conv2d
ReLU
Sequential
Conv2d
ReLU
Sequential
Conv2d
Flatten
Sequential


In [49]:
def init_weights(m):
    if isinstance(m, (nn.Conv1d,nn.Conv2d,nn.Conv3d)): init.kaiming_normal_(m.weight)

In [50]:
model.apply(init_weights);

In [51]:
MomentumLearner(model, dls, F.cross_entropy, cbs=[DeviceCB()]).lr_find()

In [52]:
set_seed(42)
learn = MomentumLearner(get_model().apply(init_weights), dls, F.cross_entropy, lr=0.2, cbs=cbs)

In [53]:
learn.fit(3)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 509.141 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.141 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.586,1.230,0,train
0.777,0.606,0,eval
0.818,0.490,1,train
0.829,0.465,1,eval
0.843,0.424,2,train
0.853,0.412,2,eval


Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.141 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.141 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.140 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.140 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.139 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.139 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.138 GB. Budget is not Drive quota.


In [54]:
astats.color_dim()

In [55]:
astats.plot_stats()

## Input normalization

In [56]:
xmean,xstd = xb.mean(),xb.std()
xmean,xstd

(tensor(0.29), tensor(0.35))

In [57]:
#| export
class BatchTransformCB(Callback):
    def __init__(self, tfm, on_train=True, on_val=True): fc.store_attr()

    def before_batch(self, learn):
        if (self.on_train and learn.training) or (self.on_val and not learn.training):
            learn.batch = self.tfm(learn.batch)

In [58]:
def _norm(b): return (b[0]-xmean)/xstd,b[1]
norm = BatchTransformCB(_norm)

In [59]:
set_seed(42)
learn = MomentumLearner(get_model().apply(init_weights), dls, F.cross_entropy, lr=0.2, cbs=cbs+[norm])
learn.fit(3)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 509.136 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.136 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.548,1.396,0,train
0.766,0.608,0,eval
0.809,0.516,1,train
0.820,0.498,1,eval
0.843,0.431,2,train
0.841,0.439,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.136 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.136 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.136 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.136 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.136 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.135 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.135 GB. Budget is not Drive quota.


In [60]:
astats.color_dim()

In [61]:
astats.plot_stats()

In [62]:
@inplace
def transformi(b): b[xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[xl]]
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=4)
xb,yb = next(iter(dls.train))

In [63]:
xb.mean(),xb.std()

(tensor(0.01), tensor(1.00))

## General ReLU

In [64]:
#| export
class GeneralRelu(nn.Module):
    def __init__(self, leak=None, sub=None, maxv=None):
        super().__init__()
        self.leak,self.sub,self.maxv = leak,sub,maxv

    def forward(self, x): 
        x = F.leaky_relu(x,self.leak) if self.leak is not None else F.relu(x)
        if self.sub is not None: x -= self.sub
        if self.maxv is not None: x.clamp_max_(self.maxv)
        return x

In [65]:
#| export
def plot_func(f, start=-5., end=5., steps=100):
    x = torch.linspace(start, end, steps)
    plt.plot(x, f(x))
    plt.grid(True, which='both', ls='--')
    plt.axhline(y=0, color='k', linewidth=0.7)
    plt.axvline(x=0, color='k', linewidth=0.7)

In [66]:
plot_func(GeneralRelu(leak=0.1, sub=0.4))

In [67]:
def conv(ni, nf, ks=3, stride=2, act=nn.ReLU):
    res = nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2)
    if act: res = nn.Sequential(res, act())
    return res

In [68]:
def get_model(act=nn.ReLU, nfs=None):
    if nfs is None: nfs = [1,8,16,32,64]
    layers = [conv(nfs[i], nfs[i+1], act=act) for i in range(len(nfs)-1)]
    return nn.Sequential(*layers, conv(nfs[-1],10, act=None), nn.Flatten()).to(def_device)

In [69]:
#| export
def init_weights(m, leaky=0.):
    if isinstance(m, (nn.Conv1d,nn.Conv2d,nn.Conv3d)): init.kaiming_normal_(m.weight, a=leaky)

In [70]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
astats = ActivationStats(fc.risinstance(GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
iw = partial(init_weights, leaky=0.1)

In [71]:
model = get_model(act_gr).apply(iw)

In [72]:
set_seed(42)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 509.127 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.127 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.774,0.625,0,train
0.844,0.421,0,eval
0.868,0.361,1,train
0.861,0.373,1,eval
0.880,0.325,2,train
0.867,0.362,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.127 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.127 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.127 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 509.127 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.889 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.889 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.889 GB. Budget is not Drive quota.


In [73]:
astats.color_dim()

In [74]:
astats.plot_stats()

In [75]:
astats.dead_chart()

## LSUV

[All You Need is a Good Init](https://arxiv.org/pdf/1511.06422.pdf) introduces *Layer-wise Sequential Unit-Variance* (*LSUV*).

In [76]:
#| export
def _lsuv_stats(hook, mod, inp, outp):
    acts = to_cpu(outp)
    hook.mean = acts.mean()
    hook.std = acts.std()

def lsuv_init(model, m, m_in, xb):
    h = Hook(m, _lsuv_stats)
    with torch.no_grad():
        while model(xb) is not None and (abs(h.std-1)>1e-3 or abs(h.mean)>1e-3):
            m_in.bias -= h.mean
            m_in.weight.data /= h.std
    h.remove()

In [77]:
model = get_model(act_gr)
relus = [o for o in model.modules() if isinstance(o, GeneralRelu)]
convs = [o for o in model.modules() if isinstance(o, nn.Conv2d)]

In [78]:
for ms in zip(relus,convs): print(ms)

(GeneralRelu(), Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)))
(GeneralRelu(), Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)))
(GeneralRelu(), Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)))
(GeneralRelu(), Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)))


In [79]:
for ms in zip(relus,convs): lsuv_init(model, *ms, xb.to(def_device))

In [80]:
set_seed(42)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 508.879 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.879 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.772,0.649,0,train
0.835,0.455,0,eval
0.863,0.378,1,train
0.851,0.400,1,eval
0.876,0.337,2,train
0.858,0.382,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.879 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.879 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.879 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.878 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.878 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.878 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.878 GB. Budget is not Drive quota.


In [81]:
astats.plot_stats()

## Batch Normalization

Sergey Ioffe and Christian Szegedy released ["Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift"](https://arxiv.org/abs/1502.03167) in 2015, saying:

> Training Deep Neural Networks is complicated by the fact that the distribution of each layer's inputs changes during training, as the parameters of the previous layers change. This slows down the training by requiring lower learning rates and careful parameter initialization... We refer to this phenomenon as internal covariate shift, and address the problem by normalizing layer inputs.

Their proposal is:

> Making normalization a part of the model architecture and performing the normalization for each training mini-batch. Batch Normalization allows us to use much higher learning rates and be less careful about initialization.

### LayerNorm

We'll start with [layer normalization](https://arxiv.org/abs/1607.06450), a simpler technique.

In [82]:
class LayerNorm(nn.Module):
    def __init__(self, dummy, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.mult = nn.Parameter(tensor(1.))
        self.add  = nn.Parameter(tensor(0.))

    def forward(self, x):
        m = x.mean((1,2,3), keepdim=True)
        v = x.var ((1,2,3), keepdim=True)
        x = (x-m) / ((v+self.eps).sqrt())
        return x*self.mult + self.add

In [83]:
#|export
def conv(ni, nf, ks=3, stride=2, act=nn.ReLU, norm=None, bias=None):
    if bias is None: bias = not isinstance(norm, (nn.BatchNorm1d,nn.BatchNorm2d,nn.BatchNorm3d))
    layers = [nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias)]
    if norm: layers.append(norm(nf))
    if act: layers.append(act())
    return nn.Sequential(*layers)

In [84]:
#|export
def get_model(act=nn.ReLU, nfs=None, norm=None):
    if nfs is None: nfs = [1,8,16,32,64]
    layers = [conv(nfs[i], nfs[i+1], act=act, norm=norm) for i in range(len(nfs)-1)]
    return nn.Sequential(*layers, conv(nfs[-1],10, act=None, norm=False, bias=True),
                         nn.Flatten()).to(def_device)

In [85]:
set_seed(42)
model = get_model(act_gr, norm=LayerNorm).apply(iw)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 508.873 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.873 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.781,0.606,0,train
0.846,0.421,0,eval
0.864,0.370,1,train
0.860,0.377,1,eval
0.882,0.321,2,train
0.868,0.359,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.873 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.873 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.875 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.875 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.872 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.872 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.872 GB. Budget is not Drive quota.


### BatchNorm

In [86]:
class BatchNorm(nn.Module):
    def __init__(self, nf, mom=0.1, eps=1e-5):
        super().__init__()
        # NB: pytorch bn mom is opposite of what you'd expect
        self.mom,self.eps = mom,eps
        self.mults = nn.Parameter(torch.ones (nf,1,1))
        self.adds  = nn.Parameter(torch.zeros(nf,1,1))
        self.register_buffer('vars',  torch.ones(1,nf,1,1))
        self.register_buffer('means', torch.zeros(1,nf,1,1))

    def update_stats(self, x):
        m = x.mean((0,2,3), keepdim=True)
        v = x.var ((0,2,3), keepdim=True)
        self.means.lerp_(m, self.mom)
        self.vars.lerp_ (v, self.mom)
        return m,v
        
    def forward(self, x):
        if self.training:
            with torch.no_grad(): m,v = self.update_stats(x)
        else: m,v = self.means,self.vars
        x = (x-m) / (v+self.eps).sqrt()
        return x*self.mults + self.adds

In [87]:
model = get_model(act_gr, norm=BatchNorm).apply(iw)
set_seed(42)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.4, cbs=cbs)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 508.870 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.870 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.800,0.550,0,train
0.831,0.543,0,eval
0.874,0.343,1,train
0.868,0.368,1,eval
0.886,0.305,2,train
0.871,0.354,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.870 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.870 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.869 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.870 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.869 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.869 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.869 GB. Budget is not Drive quota.


![Various norms](images/norms.png)

## Towards 90%...

In [88]:
dls = DataLoaders.from_dd(tds, 256, num_workers=4)

In [89]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 508.866 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.866 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.837,0.450,0,train
0.867,0.361,0,eval
0.883,0.318,1,train
0.867,0.366,1,eval
0.896,0.280,2,train
0.882,0.320,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.866 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.865 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.865 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.865 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.864 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.864 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.864 GB. Budget is not Drive quota.


In [90]:
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.05, cbs=cbs)
learn.fit(2)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 508.860 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.860 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.917,0.225,0,train
0.896,0.284,0,eval
0.922,0.212,1,train
0.899,0.278,1,eval


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.860 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.860 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.859 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.859 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 508.859 GB. Budget is not Drive quota.


## Export -

In [91]:
# The bundled miniai package is ready to import; no export is required.

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [92]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [93]:
# NativeNext independent numerical probe.
_next.probe(globals())


__HAILONG_NATIVE_NEXT_FULL_EVIDENCE__{"contract": {"passed": true, "target": "04_fastai_part2/11_initializing.ipynb", "ordinary_fits": 9, "ordinary_epochs": 24, "ordinary_train_batches": 2296, "ordinary_valid_batches": 195, "demonstrations": 2}, "numerical_artifact_subprobe_only": true, "numerical_models": [{"cell_index": 12, "original_epochs": 1, "probe_rows": 64, "cross_entropy": 1.0072864294052124, "accuracy": 0.546875, "model_checkpoint_snapshot_equal": true, "exact_saved_inference_equal": true, "activation_statistics_retained": true, "optimizer_strict_roundtrip": true, "preserved_gradients_restored": true, "rng_validation": {"python_numpy_torch_cpu_generator_restore": true, "cuda_state_devices": 1, "cuda_validation": "saved ByteTensor structure only; no GPU generator replay claim"}}, {"cell_index": 80, "original_epochs": 3, "probe_rows": 64, "cross_entropy": 0.44994574785232544, "accuracy": 0.84375, "model_checkpoint_snapshot_equal": true, "exact_saved_inference_equal": true, "act

{'contract': {'passed': True,
  'target': '04_fastai_part2/11_initializing.ipynb',
  'ordinary_fits': 9,
  'ordinary_epochs': 24,
  'ordinary_train_batches': 2296,
  'ordinary_valid_batches': 195,
  'demonstrations': 2},
 'numerical_artifact_subprobe_only': True,
 'numerical_models': [{'cell_index': 12,
   'original_epochs': 1,
   'probe_rows': 64,
   'cross_entropy': 1.0072864294052124,
   'accuracy': 0.546875,
   'model_checkpoint_snapshot_equal': True,
   'exact_saved_inference_equal': True,
   'activation_statistics_retained': True,
   'optimizer_strict_roundtrip': True,
   'preserved_gradients_restored': True,
   'rng_validation': {'python_numpy_torch_cpu_generator_restore': True,
    'cuda_state_devices': 1,
    'cuda_validation': 'saved ByteTensor structure only; no GPU generator replay claim'}},
  {'cell_index': 80,
   'original_epochs': 3,
   'probe_rows': 64,
   'cross_entropy': 0.44994574785232544,
   'accuracy': 0.84375,
   'model_checkpoint_snapshot_equal': True,
   'exact